In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [7]:
!pip install -q -U sentence-transformers faiss-cpu accelerate langchain langchain-classic langchain-community langchain-core pypdf transformers==4.52.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 85.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 89.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.9 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are install

In [8]:
import os, requests
os.makedirs("knowledge_base", exist_ok=True)

PEP = "https://raw.githubusercontent.com/python/peps/main/peps/"
OWASP = "https://raw.githubusercontent.com/OWASP/CheatSheetSeries/master/cheatsheets/"

sources = {
    "pep8.rst": PEP + "pep-0008.rst",
    "pep20.rst": PEP + "pep-0020.rst",
    "pep257.rst": PEP + "pep-0257.rst",
    "owasp_sql_injection.md": OWASP + "SQL_Injection_Prevention_Cheat_Sheet.md",
    "owasp_input_validation.md": OWASP + "Input_Validation_Cheat_Sheet.md",
    "owasp_deserialization.md": OWASP + "Deserialization_Cheat_Sheet.md",
    "owasp_secrets.md": OWASP + "Secrets_Management_Cheat_Sheet.md",
    "owasp_os_command_injection.md": OWASP + "OS_Command_Injection_Defense_Cheat_Sheet.md",
}

for name, url in sources.items():
    r = requests.get(url, timeout=30)
    print(r.status_code, name, len(r.text))
    if r.status_code == 200:
        with open(f"knowledge_base/{name}", "w", encoding="utf-8") as f:
            f.write(r.text)

200 pep8.rst 50782
200 pep20.rst 1648
200 pep257.rst 10581
200 owasp_sql_injection.md 17589
200 owasp_input_validation.md 8233
200 owasp_deserialization.md 21955
200 owasp_secrets.md 65082
200 owasp_os_command_injection.md 11336


In [9]:
my_rules = """
## Mutable default arguments
Never use a mutable object (list, dict, set) as a default argument value. It is created once and shared between all calls.
Bad: def add(item, items=[]):
Good: def add(item, items=None): items = items or []
Severity: Medium

## Bare except
Never use a bare except: clause. It hides real errors and catches KeyboardInterrupt. Catch specific exceptions.
Bad: except: pass
Good: except ValueError as e: logger.error(e)
Severity: Medium

## Swallowing exceptions silently
Never catch an exception and ignore it with pass. Errors disappear and bugs become impossible to trace. Log or handle them.
Severity: Medium

## Use of eval and exec
Never call eval() or exec() on user input. It allows arbitrary code execution. Use ast.literal_eval for simple data.
Severity: Critical

## SQL injection
Never build SQL queries with string concatenation or f-strings. Use parameterized queries.
Bad: cursor.execute("SELECT * FROM users WHERE id=" + user_id)
Good: cursor.execute("SELECT * FROM users WHERE id=%s", (user_id,))
Severity: Critical

## Hardcoded secrets
Never hardcode passwords, API keys or tokens in source code. Load them from environment variables or a secrets manager.
Bad: API_KEY = "sk-12345"
Good: API_KEY = os.environ["API_KEY"]
Severity: High

## subprocess with shell=True
Avoid subprocess.run(cmd, shell=True) with user-controlled input. It allows command injection. Pass a list of arguments with shell=False.
Severity: High

## Unsafe deserialization with pickle
Never unpickle data from untrusted sources. pickle.loads can execute arbitrary code. Use JSON instead. Likewise use yaml.safe_load, not yaml.load.
Severity: High

## Comparing to None
Use is None or is not None, not == None. Identity checks are correct and faster.
Bad: if x == None:
Good: if x is None:
Severity: Low

## Files not closed properly
Open files with a with statement so they are always closed, even on errors.
Bad: f = open("a.txt"); data = f.read()
Good: with open("a.txt") as f: data = f.read()
Severity: Medium

## Unused imports and variables
Remove imports and variables that are never used. They clutter the code and confuse readers.
Severity: Low

## Naming conventions
Use snake_case for functions and variables, PascalCase for classes and UPPER_CASE for constants. Avoid meaningless names like x1, data2, temp.
Severity: Low

## Missing docstrings
Public modules, functions, classes and methods should have docstrings that explain what they do, their parameters and their return value.
Severity: Low

## Modifying a list while iterating over it
Do not add or remove items from a list while looping over it. It skips elements. Iterate over a copy or build a new list with a comprehension.
Severity: Medium
"""

with open("knowledge_base/my_rules.md", "w", encoding="utf-8") as f:
    f.write(my_rules)

print(os.listdir("knowledge_base"))

['pep257.rst', 'owasp_input_validation.md', 'pep8.rst', 'pep20.rst', 'my_rules.md', 'owasp_sql_injection.md', 'owasp_os_command_injection.md', 'owasp_secrets.md', 'owasp_deserialization.md']


In [10]:
from sentence_transformers import SentenceTransformer
import faiss

def load_rules(path):
    # each rule starts with "## ", so one rule = one chunk
    text = open(path, encoding="utf-8").read()
    parts = text.split("## ")
    return [p.strip() for p in parts if p.strip()]

def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

def embed_chunks(chunks, model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, convert_to_numpy=True)
    return model, embeddings

def create_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index

def search_index(query, model, index, chunks, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]

In [11]:
chunks = []

# 1. your own rules: one chunk per rule
for rule in load_rules("knowledge_base/my_rules.md"):
    chunks.append("[my_rules] " + rule)
n_rules = len(chunks)

# 2. the other documents: split by words
for fname in sorted(os.listdir("knowledge_base")):
    if fname == "my_rules.md":
        continue
    text = open(f"knowledge_base/{fname}", encoding="utf-8").read()
    text = text.replace("\xa0", " ")
    for c in chunk_text(text):
        chunks.append(f"[{fname}] " + c)

print("Rule chunks:", n_rules)
print("Total chunks:", len(chunks))

# 3. embed and index
model_embeddings, embeddings = embed_chunks(chunks)
index = create_faiss_index(embeddings)
print("Embeddings shape:", embeddings.shape)

Rule chunks: 14
Total chunks: 231


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings shape: (231, 384)


In [12]:
test_queries = [
    "def add(item, items=[]):",
    "cursor.execute('SELECT * FROM users WHERE id=' + user_id)",
    "try:\n    risky()\nexcept:\n    pass",
    "password = 'admin123'",
    "result = eval(user_input)",
]

for q in test_queries:
    print("\nQUERY:", q)
    for chunk in search_index(q, model_embeddings, index, chunks, k=3):
        print("  -", chunk[:100].replace("\n", " "))


QUERY: def add(item, items=[]):
  - [pep8.rst] other contexts. - Always use a def statement instead of an assignment statement that bind
  - [my_rules] Mutable default arguments Never use a mutable object (list, dict, set) as a default argum
  - [pep8.rst] foo(x): if x >= 0: return math.sqrt(x) def bar(x): if x < 0: return return math.sqrt(x) -

QUERY: cursor.execute('SELECT * FROM users WHERE id=' + user_id)
  - [my_rules] Use of eval and exec Never call eval() or exec() on user input. It allows arbitrary code 
  - [my_rules] SQL injection Never build SQL queries with string concatenation or f-strings. Use paramet
  - [owasp_os_command_injection.md] execute other commands. It is always preferable to use `escapeshella

QUERY: try:
    risky()
except:
    pass
  - [pep8.rst] (bare except is equivalent to ``except BaseException:``). A good rule of thumb is to limi
  - [owasp_sql_injection.md] Sheet](Input_Validation_Cheat_Sheet.md). ### Defense Option 4: STRONGLY DIS
  - [my_rules] Swal

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

used = sum(torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count()))
print("GPU memory used:", round(used / 1e9, 1), "GB")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

GPU memory used: 25.0 GB


In [14]:
def generate_text(prompt, max_new_tokens=700, temperature=0.3):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [15]:
print(generate_text("Explain in two sentences what a mutable default argument in Python is."))

 A mutable default argument is a default argument in Python that is mutable, meaning it can be changed after it has been created. This can lead to unexpected behavior, as the same object is used for each function call with the default argument, and changes to it persist between calls.


In [16]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import re

issues_schema = ResponseSchema(
    name="issues",
    description="A list of dictionaries with keys: file, issue, severity (one of Low, Medium, High, Critical), explanation, suggested_fix"
)

output_parser = StructuredOutputParser.from_response_schemas([issues_schema])
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"issues": string  // A list of dictionaries with keys: file, issue, severity (one of Low, Medium, High, Critical), explanation, suggested_fix
}
```


In [17]:
def extract_json_block(text):
    matches = re.findall(r'```json\s*(.*?)\s*```', text, re.DOTALL)
    if matches:
        return f"```json\n{matches[-1]}\n```"
    # fallback: the model forgot the ```json fence
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        return None
    return f"```json\n{text[start:end + 1]}\n```"

def parse_issues(text):
    block = extract_json_block(text)
    if block is None:
        return []
    try:
        return output_parser.parse(block)["issues"]
    except Exception:
        return []

In [18]:
analyze_template = """
You are a senior Python developer.
Read the following Python code and describe in 3 to 5 short sentences what it does.

Code:
{code}

Summary:
"""

def analyze_code(code):
    prompt = PromptTemplate(
        template=analyze_template,
        input_variables=["code"]
    ).format(code=code)
    return generate_text(prompt, max_new_tokens=200).strip()

In [33]:
# --- RAG: a separate small index that contains only my own rules ---
rules_list = load_rules("knowledge_base/my_rules.md")
rule_embeddings = model_embeddings.encode(rules_list, convert_to_numpy=True)
rules_index = create_faiss_index(rule_embeddings)

def get_rules(code, k=3):
    found = []
    for line in code.split("\n"):
        if not line.strip():
            continue
        for r in search_index(line, model_embeddings, rules_index, rules_list, k=k):
            if r not in found:
                found.append(r)
    return found

def get_title(rule):
    return rule.split("\n")[0].strip()

def get_severity(rule):
    m = re.search(r"Severity:\s*(\w+)", rule)
    return m.group(1) if m else "Medium"

# --- Chain 2: check one rule at a time ---
check_template = """
You are a Python code reviewer.

Rule:
{rule}

Code:
{code}

Does the code contain the exact thing this rule talks about, and break the rule?
If yes, answer in this format: YES | the exact line of code, copied from the code
If no, answer exactly: NO

Answer:
"""

def detect_issues(code, rules):
    detected = []
    used_lines = []
    code_clean = " ".join(code.split())

    for rule in rules:
        prompt = PromptTemplate(
            template=check_template,
            input_variables=["rule", "code"]
        ).format(rule=rule, code=code)
        answer = generate_text(prompt, max_new_tokens=60).strip()

        if not answer.upper().startswith("YES"):
            print(f"  {get_title(rule)}: no")
            continue

        line = answer.split("|", 1)[1].strip() if "|" in answer else ""
        line = re.sub(r"^line\s*\d+\s*:\s*", "", line, flags=re.I).strip("` ")
        line_clean = " ".join(line.split())

        # check 1: the line must really exist in the code
        if not line_clean or line_clean not in code_clean:
            print(f"  {get_title(rule)}: YES but skipped (line not found in the code)")
            continue
        # check 2: do not report the same line twice
        if line_clean in used_lines:
            print(f"  {get_title(rule)}: YES but skipped (line already reported)")
            continue

        used_lines.append(line_clean)
        print(f"  {get_title(rule)}: YES")
        detected.append({
            "title": get_title(rule),
            "severity": get_severity(rule),
            "line": line,
            "rule": rule,
        })
    return detected

In [34]:
# --- RAG over the documents: a separate index with PEP 8, PEP 257, OWASP... ---
doc_ids = [i for i, c in enumerate(chunks) if not c.startswith("[my_rules]")]
doc_chunks = [chunks[i] for i in doc_ids]
docs_index = create_faiss_index(embeddings[doc_ids])
print("Document chunks indexed:", len(doc_chunks))

def docs_for_problem(d, k=2):
    refs = search_index(f"{d['title']} {d['line']}", model_embeddings, docs_index, doc_chunks, k=k)
    names = [r.split("]")[0][1:] for r in refs]
    print(f"  documents for '{d['title']}': {names}")
    return "\n".join(r.split("] ", 1)[1][:500] for r in refs)

suggest_template = """
You are a Python code reviewer.
Write ONE review item for the problem below, with: file, issue, severity, explanation and suggested_fix.
- explanation: one or two sentences about why this is a problem in this code. Use the Documentation if it is about the same problem (for example mention PEP 8 or OWASP). Ignore documentation that is not about the problem.
- suggested_fix: the corrected Python code for the problem line. Follow the "Good" example in the rule when there is one.

File name: {file_name}

Code:
{code}

Rule:
{rule}

Documentation:
{docs_text}

Problem:
{title} | {severity} | {line}

Respond ONLY in JSON format as follows. The list must contain exactly one dictionary.
{format_instructions}
"""

def suggest_improvements(code, file_name, detected):
    problems = "\n".join(f"- {d['title']} | {d['severity']} | {d['line']}" for d in detected)
    issues = []
    raws = []

    for d in detected:
        docs_text = docs_for_problem(d)
        prompt = PromptTemplate(
            template=suggest_template,
            input_variables=["file_name", "code", "rule", "docs_text", "title",
                             "severity", "line", "format_instructions"]
        ).format(file_name=file_name, code=code, rule=d["rule"], docs_text=docs_text,
                 title=d["title"], severity=d["severity"], line=d["line"],
                 format_instructions=format_instructions)
        raw = generate_text(prompt, max_new_tokens=500)
        raws.append(raw)

        parsed = parse_issues(raw)
        first = parsed[0] if parsed and isinstance(parsed[0], dict) else {}

        # title and severity come from the rules, so they are always exact
        issues.append({
            "file": file_name,
            "issue": d["title"],
            "severity": d["severity"],
            "explanation": first.get("explanation", "Could not generate an explanation."),
            "suggested_fix": first.get("suggested_fix", ""),
        })

    return problems, "\n\n".join(raws), issues

Document chunks indexed: 217


In [35]:
def review_code(code, file_name="main.py"):
    summary = analyze_code(code)                         # chain 1
    rules = get_rules(code)                              # RAG
    print("Rules found by RAG:", len(rules))
    detected = detect_issues(code, rules)                # chain 2

    if not detected:
        return {"summary": summary, "problems": "No issues found.", "issues": [], "raw": ""}

    problems, raw, issues = suggest_improvements(code, file_name, detected)   # chain 3
    return {"summary": summary, "problems": problems, "issues": issues, "raw": raw}

In [36]:
sample_code = '''import pickle
import sqlite3

password = "admin123"

def get_user(user_id, cache=[]):
    conn = sqlite3.connect("app.db")
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users WHERE id=" + user_id)
    try:
        data = pickle.loads(open("data.pkl", "rb").read())
    except:
        pass
    cache.append(user_id)
    return cursor.fetchall()
'''

result = review_code(sample_code, file_name="sample.py")

print("\nPROBLEMS FOUND:\n", result["problems"])
print("\nFINAL ISSUES:")
display(pd.DataFrame(result["issues"]))

Rules found by RAG: 11
  Unsafe deserialization with pickle: YES
  Unused imports and variables: YES
  Mutable default arguments: YES
  SQL injection: YES
  Hardcoded secrets: YES
  Use of eval and exec: no
  Bare except: YES
  subprocess with shell=True: no
  Comparing to None: YES but skipped (line not found in the code)
  Swallowing exceptions silently: YES but skipped (line already reported)
  Missing docstrings: YES
  documents for 'Unsafe deserialization with pickle': ['owasp_deserialization.md', 'owasp_deserialization.md']
  documents for 'Unused imports and variables': ['pep8.rst', 'pep8.rst']
  documents for 'Mutable default arguments': ['pep8.rst', 'pep8.rst']
  documents for 'SQL injection': ['owasp_sql_injection.md', 'owasp_sql_injection.md']
  documents for 'Hardcoded secrets': ['owasp_secrets.md', 'owasp_secrets.md']
  documents for 'Bare except': ['pep8.rst', 'pep8.rst']
  documents for 'Missing docstrings': ['owasp_os_command_injection.md', 'pep257.rst']

PROBLEMS FOUND

,file,issue,severity,explanation,suggested_fix
0,sample.py,Unsafe deserialization with pickle,High,The use of pickle.loads to deserialize data fr...,Replace pickle.loads with a safer deserializat...
1,sample.py,Unused imports and variables,Low,The 'pickle' module is imported but never used...,Remove the import statement for the 'pickle' m...
2,sample.py,Mutable default arguments,Medium,The default value for the 'cache' argument in ...,"def get_user(user_id, cache=None):\n cache ..."
3,sample.py,SQL injection,Critical,The SQL query is constructed using string conc...,"cursor.execute(""SELECT * FROM users WHERE id=?..."
4,sample.py,Hardcoded secrets,High,The password 'admin123' is hardcoded in the so...,password = os.environ['PASSWORD']
5,sample.py,Bare except,Medium,The bare except clause in the get_user functio...,except Exception as e:\n logger.error(e)
6,sample.py,Missing docstrings,Low,The function `get_user` is missing a docstring...,Add a docstring to the `get_user` function exp...


In [37]:
!pip install -q fastapi uvicorn pyngrok python-multipart

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [39]:
import threading
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))

app = FastAPI(title="Code Review API")
gpu_lock = threading.Lock()   # one review at a time

class ReviewRequest(BaseModel):
    code: str
    file_name: str = "main.py"

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/review")
def review(req: ReviewRequest):
    with gpu_lock:
        result = review_code(req.code[:6000], file_name=req.file_name)
    return {"summary": result["summary"], "issues": result["issues"]}

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()

tunnel = ngrok.connect(8000)
print("=" * 60)
print("PUBLIC URL (paste this into Streamlit):")
print(tunnel.public_url)
print("=" * 60)

PUBLIC URL (paste this into Streamlit):
https://twerp-delouse-willing.ngrok-free.dev


In [40]:
import requests

test = {"code": "password = 'admin123'\n", "file_name": "test.py"}
r = requests.post("http://localhost:8000/review", json=test, timeout=300)
print(r.status_code)
print(r.json())

Rules found by RAG: 3
  Hardcoded secrets: YES
  Use of eval and exec: no
  Bare except: no
  documents for 'Hardcoded secrets': ['owasp_secrets.md', 'owasp_secrets.md']
200
{'summary': "This Python code sets a string variable named 'password' to the value 'admin123'.", 'issues': [{'file': 'test.py', 'issue': 'Hardcoded secrets', 'severity': 'High', 'explanation': 'Hardcoding passwords in source code can lead to unauthorized access if the code is compromised.', 'suggested_fix': "password = os.environ['PASSWORD']"}]}


In [ ]:
import time

while True:
    time.sleep(60)